# NB02 — The Oxygen Attack on Ethylene: Why Transition States Are Multireference

The reaction of atomic oxygen O(³P) with ethylene provides a prototypical example where the electronic structure changes qualitatively along a reaction coordinate. In the transition-state region, the system develops pronounced biradical character, and single-reference methods become inadequate.

## The reaction
We consider the attack of atomic oxygen O(³P) on the C=C double bond of ethylene.
$$
\mathrm{O}(^3P) + \mathrm{C_2H_4} \rightarrow \text{triplet biradical intermediate}
$$

![Reaction scheme](figures/o3p_ethylene_scheme.svg)

As the oxygen atom approaches the π bond, electron pairing in the double bond is progressively weakened. The system evolves from a closed-shell π-bonded structure toward a configuration in which two unpaired electrons are distributed over the O–C–C framework.

In this regime, multiple electronic configurations become nearly degenerate, and a single determinant is no longer sufficient to describe the wavefunction.

We will build the potential energy surface step by step. Look at the curves before reading the explanation.


## 0. Setup (run this first)
This cell loads the required libraries and configures paths. It is not part of the scientific content — run it once before proceeding.


In [ ]:
import sys
sys.path.insert(0, '../tools')

import shutil
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

from utils import (
    ORCA, NPROCS, HARTREE_TO_KJMOL,
    setup_workdir, run_orca,
    get_energy, get_nevpt2_energy,
    get_distance, terminated_normally,
    get_no_occupations, plot_orbital, show_orbital
)
from qctools import (
    load_xyz_as_traj, build_xyz_trajectory,
    run_casscf_scan, run_nevpt2_scan, run_b3lyp_scan,
)

# O-C1 distance extractor — atoms 0 and 2 in the xyz files
get_oc_distance = lambda f: get_distance(f, 0, 2)

# Include %pal block only when more than one core is available

# ORCA parallelism — default serial for this small system
# Increase NPROCS_ORCA to experiment (max: NPROCS available cores)
NPROCS_ORCA = 1

print(f'ORCA:        {ORCA}')
print(f'Available:   {NPROCS} cores')
print(f'Using:       {NPROCS_ORCA} core(s) for ORCA')


## 1. Working directory
Calculation files will be written to `ethylene/`. Set `FORCE_CLEAN = True` to delete and restart from scratch — useful if you want to rerun everything cleanly.


In [ ]:
WORKDIR = 'ethylene'
FORCE_CLEAN = False  # set True to start from scratch

if FORCE_CLEAN and Path(WORKDIR).exists():
    shutil.rmtree(WORKDIR)
    print(f'Removed {WORKDIR}/')

work_dir = setup_workdir(WORKDIR)
print(f'Working in: {work_dir}')

## 2. Starting geometry

O(³P) placed at 3.50 Å from C1 along a standard approach vector; ethylene at equilibrium geometry.
Charge 0, multiplicity 3.


In [ ]:
C1 = np.array([ 0.000,  0.000,  0.000])
C2 = np.array([ 1.335,  0.000,  0.000])
O0 = np.array([-0.517,  0.000,  1.932])   # R(O-C1) = 3.50 Å

ts_geometry = """\
C    0.000000    0.000000    0.000000
C    1.335000    0.000000    0.000000
O   -0.517000    0.000000    1.932000
H   -0.585000    0.925000   -0.150000
H   -0.585000   -0.925000   -0.150000
H    1.920000    0.925000    0.050000
H    1.920000   -0.925000    0.050000
"""


The ball-and-stick view below shows the starting configuration; O is well outside bonding distance and the π bond is intact.


In [ ]:
import py3Dmol

xyz_str = f'7\nO(3P) + ethylene  R(O-C1) = 3.50 A\n{ts_geometry.strip()}'
view = py3Dmol.view(width=500, height=400)
view.addModel(xyz_str, 'xyz')
view.setStyle({'stick':  {'colorscheme': 'grayCarbon', 'radius': 0.12},
               'sphere': {'colorscheme': 'grayCarbon', 'radius': 0.30}})

# Atom labels — indices match ts_geometry order: C1=0, C2=1, O=2
atom_labels = [
    ('C1', [ 0.000,  0.000,  0.000]),
    ('C2', [ 1.335,  0.000,  0.000]),
    ('O',  [-0.517,  0.000,  1.932]),
]
for lbl, pos in atom_labels:
    view.addLabel(lbl, {
        'position': {'x': pos[0], 'y': pos[1] + 0.4, 'z': pos[2]},
        'fontSize': 13,
        'fontColor': 'black',
        'backgroundColor': 'white',
        'backgroundOpacity': 0.6,
        'showBackground': True,
    })

# Dashed O-C1 bond showing the reaction coordinate
O_pos  = [-0.517, 0.000, 1.932]
C1_pos = [ 0.000, 0.000, 0.000]
view.addCylinder({
    'start': {'x': O_pos[0],  'y': O_pos[1],  'z': O_pos[2]},
    'end':   {'x': C1_pos[0], 'y': C1_pos[1], 'z': C1_pos[2]},
    'radius': 0.04,
    'dashed': True,
    'color': 'gray',
    'opacity': 0.8,
})

# R label at midpoint of O-C1 vector
mid = [(O_pos[i] + C1_pos[i]) / 2 for i in range(3)]
view.addLabel('R = 3.50 Å', {
    'position': {'x': mid[0] + 0.3, 'y': mid[1] + 0.3, 'z': mid[2]},
    'fontSize': 14,
    'fontColor': 'gray',
    'backgroundColor': 'white',
    'backgroundOpacity': 0.7,
    'showBackground': True,
})

view.zoomTo()
view.show()


## 3. Active space selection

CASSCF requires us to choose a subset of orbitals — the *active space* — in which the wavefunction is written as a superposition of electron configurations. Too small an active space misses essential multireference character; too large is computationally impractical.

### 3a. Motivation from chemical intuition

The guiding principle is to include orbitals whose occupation changes significantly along the reaction coordinate. For O(³P) + ethylene, the chemically relevant orbitals are straightforward to identify:

- O(³P) carries two singly occupied 2p orbitals (the unpaired electrons of the triplet state)
- The C=C π bond and its antibonding partner π* are directly involved in bond breaking
- As O approaches C1, a new O–C σ bond forms, requiring a σ/σ* pair in the active space

This gives **6 electrons in 6 orbitals** — CASSCF(6,6) — by simple electron counting, before any calculation is run.

We verify this choice below by inspecting the natural orbital occupation numbers (NOONs) and shapes
at four geometries: R=2.2, 1.9, 1.7, and 1.5 Å, stepping from the entrance channel to the biradical region.
Geometries are unrelaxed — ethylene frozen, O translated linearly toward C1.


📝 **Task 1:** Based on the orbital analysis above, fill in `nel` and `norb` in the cell below before running it.

In [ ]:
# Diagnostic geometries at R = 2.2, 1.9, 1.7, 1.5 Å
# Ethylene frozen; O translated linearly toward C1
vec  = C1 - O0
R0   = np.linalg.norm(vec)
uvec = vec / R0

ethylene_tail = """H   -0.585000    0.925000   -0.150000
H   -0.585000   -0.925000   -0.150000
H    1.920000    0.925000    0.050000
H    1.920000   -0.925000    0.050000"""

def make_geom(R):
    O_pos = O0 + (R0 - R) * uvec
    return (f"C    {C1[0]:.6f}    {C1[1]:.6f}    {C1[2]:.6f}\n"
            f"C    {C2[0]:.6f}    {C2[1]:.6f}    {C2[2]:.6f}\n"
            f"O    {O_pos[0]:.6f}    {O_pos[1]:.6f}    {O_pos[2]:.6f}\n"
            f"{ethylene_tail}"), O_pos

R_diag  = [2.2, 1.9, 1.7, 1.5]
tags_diag = [f'noon_diag_{str(R).replace(".", "p")}' for R in R_diag]
geoms_diag = []
for R in R_diag:
    geom, O_pos = make_geom(R)
    geoms_diag.append(geom)
    print(f'R = {R:.1f} A  O at ({O_pos[0]:.3f}, {O_pos[1]:.3f}, {O_pos[2]:.3f})')


In [ ]:
# CASSCF(6,6) single points at R = 2.2, 1.9, 1.7, 1.5 Å
# Warm-started: each step passes .gbw to next for orbital continuity
prev_gbw_diag = None

for tag, geom, R in zip(tags_diag, geoms_diag, R_diag):
    gbw_out = (work_dir / tag).with_suffix('.gbw').resolve()
    moread  = f'! MORead\n%moinp "{prev_gbw_diag}"\n\n' if prev_gbw_diag else ''

    # Skip if already successfully completed
    out_file = work_dir / f'{tag}.out'
    if out_file.exists() and terminated_normally(out_file):
        print(f'R={R:.1f} A ... skipped (already done)')
        if gbw_out.exists():
            prev_gbw_diag = gbw_out
        continue


    inp = f"""{moread}! CASSCF cc-pVDZ TightSCF

%casscf
  nel FIXME    # 📝 Task 1: how many electrons in the active space?
  norb FIXME   # 📝 Task 1: how many orbitals?
  mult 3
  nroots 1
  SwitchStep 0.0
  MaxIter 500
  GTol 5e-3
end

* xyz 0 3
{geom}
*
"""
    print(f'Running CASSCF(6,6) at R={R:.1f} A ...')
    run_orca(tag, inp, work_dir, nprocs=NPROCS_ORCA)
    if gbw_out.exists():
        prev_gbw_diag = gbw_out
print('Done.')


### 3b. Verification: natural orbital evolution R=2.2 → 1.9 → 1.7 → 1.5 Å

The diagnostic CASSCF(6,6) single points confirm that the (6,6) active space was well-chosen.
The table below shows how each active orbital evolves along the reaction coordinate,
derived from Loewdin orbital compositions and NOON values in the ORCA output.
Each row shows one orbital tracked across four geometries — the continuous evolution
makes index reordering immediately visible as a sudden shape change between adjacent columns.

| Orbital character at R=2.2 Å | Orbital character at R=1.5 Å | Physical evolution |
|------------------------------|-----------------------------|--------------------|
| π (C=C), symmetric, occ ≈ 1.91 | mixed π/O bonding, occ ≈ 2.0 | π develops into a 3-centre π–O interaction as O approaches C1, characteristic of O(³P) + alkene reactions |
| π* (C=C), antisymmetric, occ ≈ 0.09 | O lone pair, occ ≈ 0.01 (NO reordering artifact) | π* mixes into the C2-centred SOMO (row 5); the near-empty orbital shown is an O lone pair that swapped index position |
| p (O, in-plane) SOMO, occ ≈ 1.0 | σ(O–C) bond, occ ≈ 2.0 | in-plane O lone pair rotates into the reaction plane and forms the O–C σ bond |
| p (O, out-of-plane) lone pair, occ ≈ 2.0 | O-centred SOMO, occ ≈ 1.0 | doubly occupied O lone pair becomes the persistent O-centred biradical SOMO |
| O p / σ-mixed SOMO, occ ≈ 1.0 | C2-centred SOMO, occ ≈ 1.0 | SOMO migrates from oxygen to C2, defining the carbon radical centre |
| σ*(O–C), occ ≈ 0.003 | σ*(O–C), occ ≈ 0.003 | antibonding spectator — remains essentially empty |

Although σ* remains essentially unoccupied and could be omitted from a minimal active space,
we retain the σ/σ* pair for consistency. σ* contributes only dynamic correlation and does
not participate in the multireference character, but omitting one member of a bonding/antibonding
pair introduces an undesirable asymmetry.

The three key orbital transformations visible in the gallery below:

1. **Lone pair → σ bond:** The O in-plane p SOMO (occ ≈ 1.0 at R=2.2 Å) rotates into the
   reaction plane and becomes the O–C σ bond (occ ≈ 2.0 at R=1.5 Å).

2. **Persistent O-centred SOMO:** The doubly occupied O out-of-plane lone pair (occ ≈ 2.0 at R=2.2 Å)
   loses one electron as the σ bond forms, becoming the persistent O-centred biradical SOMO (occ ≈ 1.0).

3. **SOMO migration O → C2:** A third O-centred orbital (occ ≈ 1.0 at R=2.2 Å) migrates to C2
   as the C=C bond breaks, forming the C2-centred SOMO of the triplet biradical.

The π system evolves from a symmetric C1–C2 bond toward a 3-centre π–O interaction.
Orbitals are tracked by the same MO index across all four geometries.
Note: the sign (colour) of each orbital lobe is arbitrary and may flip
between panels — this is a mathematical artefact with no physical meaning.
Only the shape and the occupation number carry physical information.


In [ ]:
import gc
import py3Dmol
from IPython.display import HTML

# MO indices at each R value for each orbital row
# Format: (label, [mo_idx_2p2, mo_idx_1p9, mo_idx_1p7, mo_idx_1p5], change_type)
# Indices at R=1.9 and R=1.7 are provisional — verify visually from rendered gallery
rows = [
    ('pi(C=C)',              [10, 10, 10, 10], 'real'    ),
    ('pi*(C=C)',             [13, 13, 13, 13], 'artifact'),
    ('p(O, in-plane) SOMO',  [11, 11, 11,  9], 'real'    ),
    ('p(O, out-of-plane)',   [ 9,  9,  9, 12], 'real'    ),
    ('O p/sigma-mix SOMO',   [12, 12, 12, 11], 'real'    ),
    ('sigma*(O-C)',          [14, 14, 14, 14], None      ),
]

occs_all = [
    get_no_occupations(work_dir / f'{tag}.out')[-6:]
    for tag in tags_diag
]

change_html = {
    'real':     '<span style="color:#27ae60; font-size:11px">&#9654; real</span>',
    'artifact': '<span style="color:#e74c3c; font-size:11px">&#9888; NO reordering</span>',
    None:       '',
}

col_colors = ['#2980b9', '#1a6b3a', '#8e44ad', '#c0392b']



In [ ]:
# Orbital row 1: pi(C=C)
lbl, mo_indices, change = rows[0]

# Collect cubes for this orbital row
cubes = []
for col, (tag, mo_idx) in enumerate(zip(tags_diag, mo_indices)):
    cube = plot_orbital(tag, mo_index=mo_idx, work_dir=work_dir, resolution=30)
    cubes.append(cube)
# Header row
header_cells = ''.join(
    f'<div style="width:200px; text-align:center; color:{col_colors[j]}; font-size:12px;">'
    f'<b>{lbl}</b><br>R={R_diag[j]:.1f} &#8491; occ={occs_all[j][mo_indices[j]-9]:.3f}</div>'
    for j in range(4)
)
flag = change_html[change]
display(HTML(
    f'<div style="display:flex; width:850px; font-family:sans-serif; '
    f'margin-top:18px; margin-bottom:2px; align-items:center;">'
    f'{header_cells}'
    f'<div style="width:50px; text-align:center">{flag}</div>'
    f'</div>'
))

# 4-panel orbital viewer with phase-aligned cubes
view = py3Dmol.view(width=800, height=220, linked=False, viewergrid=(1, 4))
for col, cube in enumerate(cubes):
    view.addVolumetricData(cube, 'cube',
                           {'isoval':  0.06, 'color': 'blue', 'opacity': 0.75},
                           viewer=(0, col))
    view.addVolumetricData(cube, 'cube',
                           {'isoval': -0.06, 'color': 'red',  'opacity': 0.75},
                           viewer=(0, col))
    view.addModel(cube, 'cube', viewer=(0, col))
    view.setStyle({'stick': {'colorscheme': 'grayCarbon', 'radius': 0.08}},
                  viewer=(0, col))
view.zoomTo()
view.show()

del cubes, view
gc.collect()


In [ ]:
# Orbital row 2: pi*(C=C)
lbl, mo_indices, change = rows[1]

# Collect cubes for this orbital row
cubes = []
for col, (tag, mo_idx) in enumerate(zip(tags_diag, mo_indices)):
    cube = plot_orbital(tag, mo_index=mo_idx, work_dir=work_dir, resolution=30)
    cubes.append(cube)
# Header row
header_cells = ''.join(
    f'<div style="width:200px; text-align:center; color:{col_colors[j]}; font-size:12px;">'
    f'<b>{lbl}</b><br>R={R_diag[j]:.1f} &#8491; occ={occs_all[j][mo_indices[j]-9]:.3f}</div>'
    for j in range(4)
)
flag = change_html[change]
display(HTML(
    f'<div style="display:flex; width:850px; font-family:sans-serif; '
    f'margin-top:18px; margin-bottom:2px; align-items:center;">'
    f'{header_cells}'
    f'<div style="width:50px; text-align:center">{flag}</div>'
    f'</div>'
))

# 4-panel orbital viewer with phase-aligned cubes
view = py3Dmol.view(width=800, height=220, linked=False, viewergrid=(1, 4))
for col, cube in enumerate(cubes):
    view.addVolumetricData(cube, 'cube',
                           {'isoval':  0.06, 'color': 'blue', 'opacity': 0.75},
                           viewer=(0, col))
    view.addVolumetricData(cube, 'cube',
                           {'isoval': -0.06, 'color': 'red',  'opacity': 0.75},
                           viewer=(0, col))
    view.addModel(cube, 'cube', viewer=(0, col))
    view.setStyle({'stick': {'colorscheme': 'grayCarbon', 'radius': 0.08}},
                  viewer=(0, col))
view.zoomTo()
view.show()

del cubes, view
gc.collect()


In [ ]:
# Orbital row 3: p(O, in-plane) SOMO
lbl, mo_indices, change = rows[2]

# Collect cubes for this orbital row
cubes = []
for col, (tag, mo_idx) in enumerate(zip(tags_diag, mo_indices)):
    cube = plot_orbital(tag, mo_index=mo_idx, work_dir=work_dir, resolution=30)
    cubes.append(cube)
# Header row
header_cells = ''.join(
    f'<div style="width:200px; text-align:center; color:{col_colors[j]}; font-size:12px;">'
    f'<b>{lbl}</b><br>R={R_diag[j]:.1f} &#8491; occ={occs_all[j][mo_indices[j]-9]:.3f}</div>'
    for j in range(4)
)
flag = change_html[change]
display(HTML(
    f'<div style="display:flex; width:850px; font-family:sans-serif; '
    f'margin-top:18px; margin-bottom:2px; align-items:center;">'
    f'{header_cells}'
    f'<div style="width:50px; text-align:center">{flag}</div>'
    f'</div>'
))

# 4-panel orbital viewer with phase-aligned cubes
view = py3Dmol.view(width=800, height=220, linked=False, viewergrid=(1, 4))
for col, cube in enumerate(cubes):
    view.addVolumetricData(cube, 'cube',
                           {'isoval':  0.06, 'color': 'blue', 'opacity': 0.75},
                           viewer=(0, col))
    view.addVolumetricData(cube, 'cube',
                           {'isoval': -0.06, 'color': 'red',  'opacity': 0.75},
                           viewer=(0, col))
    view.addModel(cube, 'cube', viewer=(0, col))
    view.setStyle({'stick': {'colorscheme': 'grayCarbon', 'radius': 0.08}},
                  viewer=(0, col))
view.zoomTo()
view.show()

del cubes, view
gc.collect()


In [ ]:
# Orbital row 4: p(O, out-of-plane)
lbl, mo_indices, change = rows[3]

# Collect cubes for this orbital row
cubes = []
for col, (tag, mo_idx) in enumerate(zip(tags_diag, mo_indices)):
    cube = plot_orbital(tag, mo_index=mo_idx, work_dir=work_dir, resolution=30)
    cubes.append(cube)
# Header row
header_cells = ''.join(
    f'<div style="width:200px; text-align:center; color:{col_colors[j]}; font-size:12px;">'
    f'<b>{lbl}</b><br>R={R_diag[j]:.1f} &#8491; occ={occs_all[j][mo_indices[j]-9]:.3f}</div>'
    for j in range(4)
)
flag = change_html[change]
display(HTML(
    f'<div style="display:flex; width:850px; font-family:sans-serif; '
    f'margin-top:18px; margin-bottom:2px; align-items:center;">'
    f'{header_cells}'
    f'<div style="width:50px; text-align:center">{flag}</div>'
    f'</div>'
))

# 4-panel orbital viewer with phase-aligned cubes
view = py3Dmol.view(width=800, height=220, linked=False, viewergrid=(1, 4))
for col, cube in enumerate(cubes):
    view.addVolumetricData(cube, 'cube',
                           {'isoval':  0.06, 'color': 'blue', 'opacity': 0.75},
                           viewer=(0, col))
    view.addVolumetricData(cube, 'cube',
                           {'isoval': -0.06, 'color': 'red',  'opacity': 0.75},
                           viewer=(0, col))
    view.addModel(cube, 'cube', viewer=(0, col))
    view.setStyle({'stick': {'colorscheme': 'grayCarbon', 'radius': 0.08}},
                  viewer=(0, col))
view.zoomTo()
view.show()

del cubes, view
gc.collect()


In [ ]:
# Orbital row 5: O p/sigma-mix SOMO
lbl, mo_indices, change = rows[4]

# Collect cubes for this orbital row
cubes = []
for col, (tag, mo_idx) in enumerate(zip(tags_diag, mo_indices)):
    cube = plot_orbital(tag, mo_index=mo_idx, work_dir=work_dir, resolution=30)
    cubes.append(cube)
# Header row
header_cells = ''.join(
    f'<div style="width:200px; text-align:center; color:{col_colors[j]}; font-size:12px;">'
    f'<b>{lbl}</b><br>R={R_diag[j]:.1f} &#8491; occ={occs_all[j][mo_indices[j]-9]:.3f}</div>'
    for j in range(4)
)
flag = change_html[change]
display(HTML(
    f'<div style="display:flex; width:850px; font-family:sans-serif; '
    f'margin-top:18px; margin-bottom:2px; align-items:center;">'
    f'{header_cells}'
    f'<div style="width:50px; text-align:center">{flag}</div>'
    f'</div>'
))

# 4-panel orbital viewer with phase-aligned cubes
view = py3Dmol.view(width=800, height=220, linked=False, viewergrid=(1, 4))
for col, cube in enumerate(cubes):
    view.addVolumetricData(cube, 'cube',
                           {'isoval':  0.06, 'color': 'blue', 'opacity': 0.75},
                           viewer=(0, col))
    view.addVolumetricData(cube, 'cube',
                           {'isoval': -0.06, 'color': 'red',  'opacity': 0.75},
                           viewer=(0, col))
    view.addModel(cube, 'cube', viewer=(0, col))
    view.setStyle({'stick': {'colorscheme': 'grayCarbon', 'radius': 0.08}},
                  viewer=(0, col))
view.zoomTo()
view.show()

del cubes, view
gc.collect()


In [ ]:
# Orbital row 6: sigma*(O-C)
lbl, mo_indices, change = rows[5]

# Collect cubes for this orbital row
cubes = []
for col, (tag, mo_idx) in enumerate(zip(tags_diag, mo_indices)):
    cube = plot_orbital(tag, mo_index=mo_idx, work_dir=work_dir, resolution=30)
    cubes.append(cube)
# Header row
header_cells = ''.join(
    f'<div style="width:200px; text-align:center; color:{col_colors[j]}; font-size:12px;">'
    f'<b>{lbl}</b><br>R={R_diag[j]:.1f} &#8491; occ={occs_all[j][mo_indices[j]-9]:.3f}</div>'
    for j in range(4)
)
flag = change_html[change]
display(HTML(
    f'<div style="display:flex; width:850px; font-family:sans-serif; '
    f'margin-top:18px; margin-bottom:2px; align-items:center;">'
    f'{header_cells}'
    f'<div style="width:50px; text-align:center">{flag}</div>'
    f'</div>'
))

# 4-panel orbital viewer with phase-aligned cubes
view = py3Dmol.view(width=800, height=220, linked=False, viewergrid=(1, 4))
for col, cube in enumerate(cubes):
    view.addVolumetricData(cube, 'cube',
                           {'isoval':  0.06, 'color': 'blue', 'opacity': 0.75},
                           viewer=(0, col))
    view.addVolumetricData(cube, 'cube',
                           {'isoval': -0.06, 'color': 'red',  'opacity': 0.75},
                           viewer=(0, col))
    view.addModel(cube, 'cube', viewer=(0, col))
    view.setStyle({'stick': {'colorscheme': 'grayCarbon', 'radius': 0.08}},
                  viewer=(0, col))
view.zoomTo()
view.show()

del cubes, view
gc.collect()


<details>
<summary>Active space selection — practical criteria and orbital evolution</summary>

The labels above are assigned from Loewdin orbital compositions in the ORCA output and confirmed
against established multireference studies of O(³P) + ethylene.

**Note on orbital pairing:** The current gallery uses index-based pairing, which can be
unreliable when NO eigenvalues reorder between geometries. Even for this relatively simple
system, visual and chemical analysis alone was insufficient to catch a labeling error —
it required expert review of the raw Loewdin output. For larger or less symmetric systems,
semi-automatic methods such as AVAS become not just convenient but essential.
An overlap-based matcher (6×6 NO overlap matrix + Hungarian algorithm) is planned for
`qctools.py` to make this robust.

`SwitchStep 0.0` prevents ORCA from swapping active orbitals with inactive ones during
optimization. It does not prevent reordering *within* the active space — that is a
physical feature, not a numerical artefact.

**Validation:** Running CASSCF(8,8) and checking that the two additional orbitals remain
near occ ≈ 0 and 2 throughout confirms that (6,6) captures the essential multireference
character without unnecessary expansion.

**Semi-automatic active space selection:** AVAS (Atomic Valence Active Space) automates
the selection by projecting atomic valence orbitals onto the MO space. Available in ORCA 6.x
(`%casscf AVAS true end`). For larger systems this becomes the recommended starting point
rather than manual chemical intuition.

</details>


## 4. CASSCF relaxed scan — custom R grid

Active space: 6 electrons in 6 orbitals.

Rather than a uniform grid, we use **non-uniform spacing**: fewer points in the flat
long-range tail, more points through the barrier and product region where the energy
changes rapidly.

Each point is a constrained geometry optimisation: O–C1 fixed at the target R,
all other degrees of freedom relaxed. Points are computed inward (large R → small R)
with the converged wavefunction — orbitals and CI coefficients — passed forward at
each step via `MORead`, mirroring what ORCA does internally for its scan jobs.

`MaxIter 200` gives the CASSCF optimizer enough room to converge near the barrier.

**Note:** NEVPT2 has no analytic gradient, so geometry relaxation uses CASSCF;
NEVPT2 is applied as single-point corrections in the next section.


📝 **Task 2:** Run the CASSCF scan. Fill in the active space parameters and the R grid, then execute the cell.

In [ ]:
# Non-uniform R grid: coarse in flat tail, fine through barrier and product
R_grid = [
    3.50, 3.20, 2.90,          # tail: 3 points, ~0.3 Å spacing
    2.60, 2.40, 2.20,          # approach: 3 points, ~0.2 Å spacing
    2.05, 1.90, 1.80, 1.70,   # barrier: 4 points, ~0.1 Å spacing
    1.60, 1.50, 1.40, 1.30,   # product region: 4 points, ~0.1 Å spacing
    1.22,                      # near minimum
]  # 15 points total

casscf_inp = """\
{moread}! CASSCF {basis} TightSCF Opt

%maxcore 4000

%geom
  Constraints
    {{B 0 2 {R:.3f} C}}
  end
end

%casscf
  nel {nel}
  norb {norb}
  mult {mult}
  nroots 1
  MaxIter {maxiter}
end

* xyz 0 3
{geom}
*
"""

casscf_results = run_casscf_scan(
    R_grid, ts_geometry, work_dir, casscf_inp,
    nel=FIXME, norb=FIXME, mult=3,  # 📝 Task 2: fill in the active space
    nprocs=NPROCS_ORCA
)
R_casscf_arr = np.array([r for r, e in casscf_results])
E_casscf_arr = np.array([e for r, e in casscf_results])


## While the scan runs...

The CASSCF scan above will take a few minutes. This is a good time to read the paper
that provides the experimental and theoretical background for this reaction:

> Casavecchia et al., *Experimental and theoretical studies of the O(³P) + C₂H₄ reaction dynamics:
> collision energy dependence of branching ratios and extent of intersystem crossing*

[Download PDF](https://www.researchgate.net/profile/Piergiorgio-Casavecchia/publication/233947839_Experimental_and_theoretical_studies_of_the_O3P_C2H4_reaction_dynamics_Collision_energy_dependence_of_branching_ratios_and_extent_of_intersystem_crossing/links/53dfb7b80cf2aede4b492ce9/Experimental-and-theoretical-studies-of-the-O3P-C2H4-reaction-dynamics-Collision-energy-dependence-of-branching-ratios-and-extent-of-intersystem-crossing.pdf)

Pay particular attention to **Figure 1** — it shows the full reaction network and illustrates
why even this apparently simple system requires careful treatment of both static and dynamic
correlation. The multiple product channels, surface crossings, and collision-energy-dependent
branching ratios all emerge from the same biradical intermediate we are computing here.

Things to look for while reading:
- What products are observed and in what ratios?
- What role does intersystem crossing play?
- How does the barrier height from the paper compare to what we compute?
- Why is a multireference treatment necessary to describe this reaction correctly?

<details>
<summary>Why does this matter? Connections to atmospheric chemistry</summary>

### Atmospheric chemistry

O(³P) is produced continuously in the troposphere by UV photolysis of O₃ (λ > 310 nm)
and NO₂. Steady-state concentrations are low (~10⁴–10⁶ atoms/cm³) but chemically
significant because reactions with alkenes are fast (~10⁻¹² cm³/molecule/s).
O(¹D) — singlet atomic oxygen from O₃ photolysis at shorter wavelengths — is even
more reactive and is the primary source of tropospheric OH•, the main atmospheric oxidant.

The barrier height we compute directly enters the Arrhenius expression for the rate constant.
An error of 10 kJ/mol changes the rate by an order of magnitude at atmospheric temperatures —
which is why getting the multireference description right matters for atmospheric models.

The competing product channels visible in Figure 1 of the Casavecchia paper — which products
form and in what ratios — depend on the ISC efficiency at the biradical geometry. These
branching ratios determine the ultimate fate of the carbon in atmospheric VOC oxidation
chains, with direct consequences for ozone formation and secondary organic aerosol.

### Mechanistic analogy to condensed-phase oxidation

The O(³P) + alkene reaction is a well-studied gas-phase model for radical attack on C=C
double bonds. The same chemical topology — triplet radical approaching a π bond, biradical
intermediate, competing product channels — appears in condensed-phase oxidation chemistry.
Whether this analogy extends quantitatively to biological systems is an open question
beyond the scope of this notebook.

### Connection to biological radical chemistry

The O(³P) + alkene reaction is a gas-phase model for the class of radical initiation
reactions that ultimately produce OH• in biological systems — via the hydrogen
abstraction channel directly, or through fragmentation of the biradical addition
products. OH• is the most reactive ROS in biology: it damages DNA, lipids, and
proteins indiscriminately and is central to radiation damage, Fenton chemistry,
and ischaemia-reperfusion injury.

While the specific oxidant in vivo differs from O(³P), the underlying spin-surface
physics — a triplet radical attacking a closed-shell π system, biradical formation,
competing reaction channels — is the same class of chemistry. The O(³P) + ethylene
system is the minimal, computationally tractable model for studying it.

</details>


## 5. NEVPT2 single points

NEVPT2 adds dynamic correlation on top of the CASSCF reference.
We use the `.gbw` file from each scan step as the orbital guess via `MORead` —
CASSCF re-converges in very few iterations since the orbitals are already optimised for that geometry.


📝 **Task 3:** Run NEVPT2 single points on the CASSCF geometries.

In [ ]:
nevpt2_inp = """\
! NEVPT2 {basis} TightSCF MORead

%maxcore 4000

%moinp "{gbw_file}"

%casscf
  nel {nel}
  norb {norb}
  mult {mult}
  nroots 1
end

* xyz 0 3
{geom}
*
"""

nevpt2_results = run_nevpt2_scan(
    casscf_results, work_dir, nevpt2_inp,
    nel=6, norb=6, mult=3,        # must match CASSCF active space
    nprocs=NPROCS_ORCA
)
R_nevpt2_arr = np.array([r for r, e in nevpt2_results])
E_nevpt2_arr = np.array([e for r, e in nevpt2_results])


## 6. B3LYP single points on CASSCF geometries

B3LYP is evaluated at the same geometries as the CASSCF scan — this isolates the
effect of the method from geometry differences.
The gaps in the B3LYP curve mark exactly where the single-reference description breaks down.

<details>
<summary>Convergence strategy</summary>

Single points are computed outward (small R → large R) with the converged wavefunction
passed forward at each step via `MORead`. Starting from the product side where the triplet
is well-defined gives better convergence through the barrier region.

On a triplet surface ORCA uses ROB3LYP (restricted open-shell) by default.
Forcing unrestricted (`! UKS B3LYP`) gives identical results — the convergence
failures are physical, not a technical limitation of the restricted framework.

</details>


📝 **Task 4:** Run B3LYP single points for comparison.

In [ ]:
b3lyp_inp = """\
{moread}! B3LYP {basis} TightSCF SlowConv

%maxcore 4000

%scf
  MaxIter 500
  STABPerform true
end

* xyz 0 3
{geom}
*
"""

b3lyp_results = run_b3lyp_scan(
    casscf_results, work_dir, b3lyp_inp,
    nprocs=NPROCS_ORCA
)
R_b3lyp_arr = np.array([r for r, e in b3lyp_results])
E_b3lyp_arr = np.array([e for r, e in b3lyp_results])


## 7. Collect and normalise

All three methods share the same CASSCF-relaxed geometries.
Energies are normalised to zero at the reactant asymptote (R=3.5 Å).


In [ ]:
# All arrays sorted large R first
R_all = R_casscf_arr

E_casscf_rel = (E_casscf_arr - E_casscf_arr[0]) * HARTREE_TO_KJMOL
E_nevpt2_rel = (E_nevpt2_arr - E_nevpt2_arr[0]) * HARTREE_TO_KJMOL

b3lyp_ref   = E_b3lyp_arr[0] if np.isfinite(E_b3lyp_arr[0]) else \
               E_b3lyp_arr[np.isfinite(E_b3lyp_arr)][0]
E_b3lyp_rel = np.where(
    np.isfinite(E_b3lyp_arr),
    (E_b3lyp_arr - b3lyp_ref) * HARTREE_TO_KJMOL,
    np.nan
)

print(f'R range: {R_all.min():.2f} - {R_all.max():.2f} A  ({len(R_all)} points)')
print(f'B3LYP convergence failures: {np.isnan(E_b3lyp_arr).sum()}')


## 8. Energy profile

Three methods, same geometries, same basis set. Look at the curves first — then read the table.


📝 **Task 5:** Plot the energy profile. Compare the three methods.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(R_all, E_casscf_rel, 'o--', color='steelblue',   label='CASSCF(6,6)')
ax.plot(R_all, E_nevpt2_rel, 'o-',  color='darkorange',  label='CASSCF + NEVPT2')
ax.plot(R_all, E_b3lyp_rel,  'o-',  color='forestgreen', label='B3LYP')

ax.axhline(0, color='gray', lw=0.8, ls=':')
ax.set_xlabel('R(O–C1) [Å]', fontsize=12)
ax.set_ylabel('Relative energy [kJ/mol]', fontsize=12)
ax.set_title('O(³P) attack on ethylene: B3LYP vs CASSCF vs NEVPT2', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig(work_dir / 'final_profile.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to', work_dir / 'final_profile.png')


| Method | Barrier? | Why? |
|--------|----------|------|
| B3LYP | None | Single-reference — cannot describe the open-shell biradical character near the TS |
| CASSCF(6,6) | Yes, ~68 kJ/mol | Multireference, but missing dynamic correlation — overestimates the barrier |
| CASSCF + NEVPT2 | ~35 kJ/mol | Multireference + dynamic correlation — best estimate |

The B3LYP gaps are geometries where the single-reference wavefunction was unstable and ORCA aborted.
The difference between CASSCF and NEVPT2 quantifies the contribution of dynamic correlation to the barrier height.


## 9. Build animation trajectory


In [ ]:
# Build trajectory from final relaxed geometry at each R point
# Each casscf_R*.xyz contains the full optimization history — we want only the last frame
def extract_last_frame(xyz_file):
    """Return the last geometry from a multi-frame xyz file as a single-frame xyz string."""
    text   = Path(xyz_file).read_text()
    blocks = []
    lines  = text.splitlines()
    i = 0
    while i < len(lines):
        if lines[i].strip().isdigit():
            n = int(lines[i].strip())
            blocks.append('\n'.join(lines[i:i+n+2]))
            i += n + 2
        else:
            i += 1
    return blocks[-1] if blocks else text

# Sort by R descending (large R first = reactant → product direction)
xyz_files_sorted = sorted(
    work_dir.glob('casscf_R*.xyz'),
    key=get_oc_distance,
    reverse=True
)

# Write single-frame trajectory
traj_path = work_dir / 'full_path.xyz'
with open(traj_path, 'w') as f:
    for xyz_file in xyz_files_sorted:
        f.write(extract_last_frame(xyz_file) + '\n')

R_frames      = np.array([get_oc_distance(f) for f in xyz_files_sorted])
nevpt2_dict   = {round(r, 2): e for r, e in nevpt2_results}
nevpt2_frames = np.array([nevpt2_dict.get(round(get_oc_distance(f), 2), float('nan'))
                           for f in xyz_files_sorted])
E_frames = (nevpt2_frames - nevpt2_frames[0]) * HARTREE_TO_KJMOL

print(f'Trajectory: {len(R_frames)} frames, R = {R_frames.max():.2f} \u2192 {R_frames.min():.2f} \u00c5')


## 10. Interactive trajectory viewer

Step through frames using the nglview player controls (hover over the molecule).
The red dot on the energy curve tracks the current geometry.

Things to look for:
- Pyramidalization of C1 as O approaches
- C=C bond lengthening through the barrier region
- Geometry of the triplet biradical intermediate at short R


In [ ]:
import nglview
%matplotlib widget

traj = load_xyz_as_traj(str(traj_path), silent=True)
view = nglview.show_asetraj(traj)
view._set_size('400px', '350px')

plt.ioff()
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(R_frames, E_frames, 'o-', color='darkorange', label='NEVPT2')
marker, = ax.plot([R_frames[0]], [E_frames[0]], 'ro', ms=10)
ax.set_xlabel('R(O\u2013C1) [\u00c5]')
ax.set_ylabel('Relative energy [kJ/mol]')
ax.set_title('Click a point to show structure', fontsize=10, color='gray')
ax.grid(True, alpha=0.4)
ax.legend()
plt.tight_layout()
fig.canvas.layout = widgets.Layout(width='400px', height='350px')

def on_frame_change(change):
    """Sync marker when nglview frame changes (e.g. via player controls)."""
    i = change['new']
    marker.set_data([R_frames[i]], [E_frames[i]])
    fig.canvas.draw_idle()

def on_plot_click(event):
    """Jump to nearest R point when user clicks on the energy curve."""
    if event.inaxes != ax or event.xdata is None:
        return
    # Find index of nearest R value to click
    i = int(np.argmin(np.abs(R_frames - event.xdata)))
    # Update marker
    marker.set_data([R_frames[i]], [E_frames[i]])
    fig.canvas.draw_idle()
    # Jump nglview to corresponding frame
    view.frame = i

view.observe(on_frame_change, names=['frame'])
fig.canvas.mpl_connect('button_press_event', on_plot_click)

panel = widgets.HBox([view, fig.canvas])
display(panel)
plt.ion()


## 11. Questions for Students

1. **Why does B3LYP predict no barrier?**  
   Think about the electronic structure of O(³P) and what happens to the spin as it approaches the π system.

2. **Why does B3LYP fail to converge at several geometries in the approach region?**  
   What does a negative stability eigenvalue mean physically?

3. **Why does CASSCF overestimate the barrier?**  
   What kind of electron correlation is CASSCF missing? What does NEVPT2 add?

4. **What is the active space (6,6) capturing here?**  
   Look at the orbital gallery in section 3. Which orbitals have fractional occupations, and what does that tell you about where the multiconfigurational character is concentrated?

5. **How does the geometry change along the reaction path?**  
   Use the animation to identify when pyramidalization of C1 begins.
   What does this tell us about the timing of bond formation?

6. **How does spin density change along the reaction path?**  
   Where does the unpaired spin end up in the product?
   What kind of intermediate is formed?
